# Step 1: Data Cleaning - Car Price Prediction

In this notebook we take the raw `cars_dataset.csv` file and clean it step by step.
At the end we save a clean file called `cars_cleaned.csv`, which the next notebooks will use.

**What we will fix**
1. Remove the useless `Car_ID` column
2. Fix messy text (e.g. `audi`, `AUDI`, `Audi `)
3. Fix mixed values (`Yes/No` mixed with `1/0`)
4. Turn text columns like `2.0 Turbo` and `103577 km` into numbers
5. Fix impossible prices (`-999`)
6. Fill missing values

## 1. Import libraries

In [ ]:
pip install pandas

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the dataset

In [ ]:
df = pd.read_csv("cars_dataset.csv")
print("Rows and columns:", df.shape)
df.head()

## 3. Understand the data

In [ ]:
# Column names, data types and non-null counts
df.info()

In [ ]:
# Quick statistics of the number columns
df.describe()

Look closely: `Engine_Volume` and `Mileage` are shown as text (object/str) instead of numbers, and the minimum `Price` is **-999**, which is not a real price.

## 4. Check missing values and duplicates

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 5. Drop the `Car_ID` column
Every car has a different ID, so it does not help to predict the price.

In [ ]:
df = df.drop(columns=["Car_ID"])
df.columns.tolist()

## 6. Clean the `Manufacturer` text
The same brand is written in many ways (`audi`, `AUDI`, `Audi `). We make them all the same.

In [ ]:
print("Before:", df["Manufacturer"].nunique(), "different names")

df["Manufacturer"] = df["Manufacturer"].str.strip()   # remove extra spaces
df["Manufacturer"] = df["Manufacturer"].str.title()   # 'audi' / 'AUDI' -> 'Audi'
df["Manufacturer"] = df["Manufacturer"].replace({"Bmw": "BMW"})  # BMW should stay in capitals

print("After :", df["Manufacturer"].nunique(), "different names")
df["Manufacturer"].value_counts()

## 7. Fill missing `Manufacturer` using the `Model` column
A model always belongs to one brand (a *Q7* is always an Audi), so we can recover the missing brand.

In [ ]:
# Each model should belong to exactly one manufacturer (this should print 1)
print(df.groupby("Model")["Manufacturer"].nunique().max())

In [ ]:
# Make a small lookup table: Model -> Manufacturer
model_to_maker = (df.dropna(subset=["Manufacturer"])
                    .drop_duplicates("Model")
                    .set_index("Model")["Manufacturer"])

print("Missing before:", df["Manufacturer"].isnull().sum())
df["Manufacturer"] = df["Manufacturer"].fillna(df["Model"].map(model_to_maker))
print("Missing after :", df["Manufacturer"].isnull().sum())

## 8. Fix `Leather_Interior`
This column mixes `Yes/No` with `1/0`. We make everything `Yes/No`.

In [ ]:
print(df["Leather_Interior"].value_counts())

In [ ]:
df["Leather_Interior"] = df["Leather_Interior"].replace({"1": "Yes", "0": "No"})
df["Leather_Interior"].value_counts()

## 9. Fix `Engine_Volume`
Some values look like `2.0 Turbo`. We:
- create a new column `Turbo` (1 = turbo, 0 = not turbo)
- keep only the number in `Engine_Volume`

In [ ]:
df["Engine_Volume"].head(10)

In [ ]:
# 1) New column: is the engine turbo?
df["Turbo"] = df["Engine_Volume"].str.contains("Turbo", na=False).astype(int)

# 2) Keep only the number and convert to float
df["Engine_Volume"] = df["Engine_Volume"].str.replace("Turbo", "").str.strip().astype(float)

df[["Engine_Volume", "Turbo"]].head(10)

## 10. Fix `Mileage`
Some values look like `103577 km`. We remove the word `km` and convert to a number.

In [ ]:
df["Mileage"] = df["Mileage"].str.replace("km", "").str.strip().astype(float)
df["Mileage"].describe()

## 11. Fix `Price`
A price of **-999** is a placeholder for 'unknown'. Because `Price` is the value we want to predict, we cannot guess it, so we remove those rows.

In [ ]:
print("Rows with Price = -999:", (df["Price"] == -999).sum())
print("Rows with missing Price:", df["Price"].isnull().sum())

In [ ]:
df["Price"] = df["Price"].replace(-999, np.nan)   # treat -999 as missing
df = df.dropna(subset=["Price"])                  # remove rows without a price
df = df.reset_index(drop=True)

print("Rows left:", df.shape[0])
df["Price"].describe()

## 12. Fill the remaining missing values
- **Number columns** -> fill with the **median**
- **Text columns** -> fill with the **most common value (mode)**

In [ ]:
df.isnull().sum()

In [ ]:
# Number columns -> median
for col in ["Prod_Year", "Engine_Volume"]:
    df[col] = df[col].fillna(df[col].median())

# Text columns -> most common value
for col in ["Category", "Fuel_Type", "Gear_box_type", "Color", "Manufacturer"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Production year should be a whole number
df["Prod_Year"] = df["Prod_Year"].astype(int)

## 13. Final check

In [ ]:
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows      :", df.duplicated().sum())
print("Shape               :", df.shape)
df.info()

In [ ]:
df.head()

## 14. Save the clean data

In [ ]:
df.to_csv("cars_cleaned.csv", index=False)
print("Saved cars_cleaned.csv")

**Done!** Next: open `02_visualization.ipynb`.